# 6단계 권장 티어: 상품 단위 전역 DL 모델 (IT기기) — 채 담당

**분석 단위**: 상품 하나하나를 각자의 시계열로 보되, 도메인 내 여러 상품을 한 모델에 같이 넣어 전역학습(Ch.19). 필수 티어(도메인 집계)와 달리 개별 상품의 급변을 잡아낼 수 있다.

**대상**: 리뷰 30건 이상 + 관측기간 10개월 이상 상품만 (IT기기: 1,821개 상품 중 205개)

**모델**: `DLinearModel`(Ch.16, 가벼운 선형 신경망) vs `NHiTSModel`(Ch.13) + `QuantileRegression` 확률적 예측(Ch.18)

**조의 시행착오를 반영해 처음부터 3차 버전으로 시작**: 조가 생활·화장품에서 3번 시도 끝에 알아낸 핵심은 "그 달 실제 리뷰수를 covariate로 추가해야 한다"는 것이었다. 이게 없으면 모델이 naive baseline(마지막 값 그대로 예측)과 소수점까지 똑같이 나올 정도로 아무것도 못 배운다. 여기서는 처음부터 리뷰수 covariate를 포함해서 시작한다 (`results/reports/06_시계열_결과_조.md` 5-2절 참고).

**검증**: 각 상품 시계열의 마지막 2개월을 hold-out, baseline(도메인평균/naive) 대비 MAE로 비교.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

from darts import TimeSeries
from darts.models import DLinearModel, NHiTSModel
from darts.utils.likelihood_models.torch import QuantileRegression
from darts.dataprocessing.transformers import Scaler

DOMAIN = "IT기기"
MIN_REVIEWS = 30
MIN_MONTHS = 10
HOLDOUT = 2
INPUT_LEN = 6
OUTPUT_LEN = HOLDOUT

reviews = pd.read_parquet("../data/processed/reviews.parquet")
sub = reviews[(reviews["Domain"] == DOMAIN) & reviews["RDate_parsed"].notna()].copy()
sub["month"] = sub["RDate_parsed"].dt.to_period("M").dt.to_timestamp()

prod_stats = sub.groupby("ProductName").agg(
    n_reviews=("review_id", "count"),
    first_date=("month", "min"),
    last_date=("month", "max"),
)
prod_stats["obs_months"] = (
    (prod_stats["last_date"].dt.to_period("M").astype(int) - prod_stats["first_date"].dt.to_period("M").astype(int)) + 1
)
qualified = prod_stats[(prod_stats["n_reviews"] >= MIN_REVIEWS) & (prod_stats["obs_months"] >= MIN_MONTHS)].index.tolist()
print(f"[{DOMAIN}] 조건 만족 상품 수: {len(qualified)} (전체 {sub['ProductName'].nunique()}개 중)")

[IT기기] 조건 만족 상품 수: 205 (전체 1821개 중)


In [2]:
def build_product_series(product: str):
    """상품 하나의 월별 (neg_ratio, review_count) 시계열을 만든다. 결측월은 0으로 채운다."""
    p = sub[sub["ProductName"] == product]
    g = p.groupby("month").agg(
        total=("review_id", "count"),
        negative=("GeneralPolarity", lambda s: (s == -1).sum()),
    )
    full_idx = pd.period_range(g.index.min().to_period("M"), g.index.max().to_period("M"), freq="M").to_timestamp()
    g = g.reindex(full_idx, fill_value=0)
    g["neg_ratio"] = np.where(g["total"] > 0, g["negative"] / g["total"], 0.0)
    return g

series_data = {}
skipped = 0
for prod in qualified:
    g = build_product_series(prod)
    if len(g) < INPUT_LEN + OUTPUT_LEN:
        skipped += 1
        continue
    series_data[prod] = g
print(f"실제 학습 대상: {len(series_data)}개 (길이 부족으로 제외: {skipped}개)")

실제 학습 대상: 205개 (길이 부족으로 제외: 0개)


In [3]:
# train/holdout 분리 + darts TimeSeries 구성
train_targets, train_covs = [], []
full_targets, full_covs = [], []
holdout_actuals = {}

for prod, g in series_data.items():
    target = TimeSeries.from_series(g["neg_ratio"], freq="MS")
    cov = TimeSeries.from_series(g["total"].astype(float), freq="MS")
    full_targets.append(target)
    full_covs.append(cov)
    train_targets.append(target[:-HOLDOUT])
    train_covs.append(cov[:-HOLDOUT])
    holdout_actuals[prod] = g["neg_ratio"].values[-HOLDOUT:]

cov_scaler = Scaler()
train_covs_scaled = cov_scaler.fit_transform(train_covs)
full_covs_scaled = cov_scaler.transform(full_covs)

print(f"학습 시계열 {len(train_targets)}개 준비 완료")

학습 시계열 205개 준비 완료


In [4]:
# baseline 1: 도메인 평균(학습 구간 전체 상품의 neg_ratio 평균)으로 예측
domain_mean = np.mean([g["neg_ratio"].values[:-HOLDOUT].mean() for g in series_data.values()])
print(f"도메인 평균(학습구간 기준): {domain_mean:.4f}")

# baseline 2: naive (마지막 관측값을 그대로 다음 달들 예측치로 사용)
def compute_mae(preds_dict):
    errs = []
    for prod, pred in preds_dict.items():
        actual = holdout_actuals[prod]
        errs.append(np.abs(np.array(pred) - actual).mean())
    return np.mean(errs)

baseline_domain_preds = {p: [domain_mean] * HOLDOUT for p in series_data}
baseline_naive_preds = {p: [g["neg_ratio"].values[-HOLDOUT - 1]] * HOLDOUT for p, g in series_data.items()}

mae_domain = compute_mae(baseline_domain_preds)
mae_naive = compute_mae(baseline_naive_preds)
print(f"도메인평균 baseline MAE: {mae_domain:.4f}")
print(f"naive baseline MAE: {mae_naive:.4f}")

도메인 평균(학습구간 기준): 0.0777
도메인평균 baseline MAE: 0.1503
naive baseline MAE: 0.1421


In [5]:
dlinear = DLinearModel(
    input_chunk_length=INPUT_LEN,
    output_chunk_length=OUTPUT_LEN,
    n_epochs=75,
    batch_size=32,
    random_state=42,
    pl_trainer_kwargs={"accelerator": "cpu", "enable_progress_bar": False, "enable_model_summary": False},
)
print("DLinear 학습 시작...")
dlinear.fit(series=train_targets, past_covariates=train_covs_scaled)
print("DLinear 학습 완료, 예측 중...")
dlinear_preds_list = dlinear.predict(n=HOLDOUT, series=train_targets, past_covariates=full_covs_scaled)
dlinear_preds = {prod: pred.values().flatten() for prod, pred in zip(series_data.keys(), dlinear_preds_list)}
mae_dlinear = compute_mae(dlinear_preds)
print(f"DLinear MAE: {mae_dlinear:.4f}")

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


DLinear 학습 시작...


`Trainer.fit` stopped: `max_epochs=75` reached.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


DLinear 학습 완료, 예측 중...
DLinear MAE: 0.1424


In [6]:
nhits = NHiTSModel(
    input_chunk_length=INPUT_LEN,
    output_chunk_length=OUTPUT_LEN,
    n_epochs=75,
    batch_size=32,
    likelihood=QuantileRegression(quantiles=[0.05, 0.5, 0.95]),
    random_state=42,
    pl_trainer_kwargs={"accelerator": "cpu", "enable_progress_bar": False, "enable_model_summary": False},
)
print("NHiTS 학습 시작...")
nhits.fit(series=train_targets, past_covariates=train_covs_scaled)
print("NHiTS 학습 완료, 예측 중...")
nhits_preds_list = nhits.predict(n=HOLDOUT, series=train_targets, past_covariates=full_covs_scaled, num_samples=200)
nhits_median = {prod: pred.quantile(0.5).values().flatten() for prod, pred in zip(series_data.keys(), nhits_preds_list)}
nhits_q05 = {prod: pred.quantile(0.05).values().flatten() for prod, pred in zip(series_data.keys(), nhits_preds_list)}
nhits_q95 = {prod: pred.quantile(0.95).values().flatten() for prod, pred in zip(series_data.keys(), nhits_preds_list)}
mae_nhits = compute_mae(nhits_median)
print(f"NHiTS MAE: {mae_nhits:.4f}")

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


NHiTS 학습 시작...


`Trainer.fit` stopped: `max_epochs=75` reached.


NHiTS 학습 완료, 예측 중...


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


NHiTS MAE: 0.1239


In [7]:
print(f"{'모델':<12} {'MAE':>8} {'naive 대비 개선률':>15}")
print(f"{'도메인평균':<12} {mae_domain:>8.4f} {(1 - mae_domain/mae_naive)*100:>14.1f}%")
print(f"{'naive':<12} {mae_naive:>8.4f} {'-':>15}")
print(f"{'DLinear':<12} {mae_dlinear:>8.4f} {(1 - mae_dlinear/mae_naive)*100:>14.1f}%")
print(f"{'NHiTS':<12} {mae_nhits:>8.4f} {(1 - mae_nhits/mae_naive)*100:>14.1f}%")

모델                MAE    naive 대비 개선률
도메인평균          0.1503           -5.7%
naive          0.1421               -
DLinear        0.1424           -0.2%
NHiTS          0.1239           12.8%


## 조기탐지 후보 검증 (2단계 필터)

In [8]:
# 1단계: 검증구간 실제값이 NHiTS 90% 예측구간 상한을 넘은 경우를 후보로, 그 중 그 달 리뷰 4건 이상만 "신뢰 가능"
candidates = []
for prod, g in series_data.items():
    actual = holdout_actuals[prod]
    upper = nhits_q95[prod]
    review_counts = g["total"].values[-HOLDOUT:]
    for i in range(HOLDOUT):
        if actual[i] > upper[i]:
            candidates.append({
                "product": prod, "month_offset": i, "actual": actual[i],
                "upper_90": upper[i], "review_count": review_counts[i],
            })

cand_df = pd.DataFrame(candidates)
print(f"1차 후보(예측구간 이탈): {len(cand_df)}건")
reliable = cand_df[cand_df["review_count"] >= 4].copy()
print(f"신뢰 가능(리뷰 4건 이상): {len(reliable)}건 / 노이즈로 제외: {len(cand_df) - len(reliable)}건")

1차 후보(예측구간 이탈): 77건
신뢰 가능(리뷰 4건 이상): 31건 / 노이즈로 제외: 46건


In [9]:
# 2단계: 신뢰 가능 후보 각각에 대해, 검증구간을 뺀 그 상품의 과거 데이터만으로 자체 평균/표준편차/역대 최고치 계산
genuine_spikes = []
for _, row in reliable.iterrows():
    prod = row["product"]
    hist = series_data[prod]["neg_ratio"].values[:-HOLDOUT]
    hist_mean, hist_std, hist_max = hist.mean(), hist.std(), hist.max()
    threshold = max(hist_max, hist_mean + 1.5 * hist_std)
    if row["actual"] > threshold:
        genuine_spikes.append({**row.to_dict(), "hist_mean": hist_mean, "hist_max": hist_max, "threshold": threshold})

genuine_df = pd.DataFrame(genuine_spikes)
print(f"신뢰 가능 후보 {len(reliable)}건 중 상품 자체 역대 최고치보다도 높은 '진짜 신규 급증': {len(genuine_df)}건")
if len(genuine_df) > 0:
    print()
    print(genuine_df[["product", "actual", "hist_mean", "hist_max"]].to_string(index=False))

신뢰 가능 후보 31건 중 상품 자체 역대 최고치보다도 높은 '진짜 신규 급증': 7건

                            product   actual  hist_mean  hist_max
                  JBL CLUB PRO+ TWS 0.250000   0.011111  0.200000
삼성전자 갤럭시Z 폴드3 5G 256GB, 자급제 자급제 공기계 0.400000   0.058199  0.222222
삼성전자 갤럭시Z 폴드3 5G 256GB, 자급제 자급제 공기계 0.500000   0.058199  0.222222
       삼성전자 갤럭시탭S7 FE Wi-Fi 64GB 정품 0.250000   0.054796  0.125000
              파인뷰 파인디지털 X6 2채널 32GB 0.500000   0.088228  0.250000
   파인뷰 파인디지털 X900 파워 2채널 32GB, 무료장착 0.416667   0.088999  0.200000
   파인뷰 파인디지털 X900 파워 2채널 32GB, 무료장착 0.222222   0.088999  0.200000


## 대표 상품 시각화 + 저장

In [10]:
import os as _os
_os.makedirs("../results/figures", exist_ok=True)

# 리뷰 수가 많은 상품 3개를 대표로 뽑아 실제값 vs NHiTS 예측(90% 구간) 시각화
top3_products = sorted(series_data.keys(), key=lambda p: series_data[p]["total"].sum(), reverse=True)[:3]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, prod in zip(axes, top3_products):
    g = series_data[prod]
    full_targets_dict = dict(zip(series_data.keys(), full_targets))
    idx = list(series_data.keys()).index(prod)
    full_targets_dict[prod].plot(ax=ax, label="실제", lw=1.5)
    nhits_preds_list[idx].plot(ax=ax, label="NHiTS 예측(90% 구간)", low_quantile=0.05, high_quantile=0.95)
    ax.set_title(prod[:20], fontsize=9)
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(f"../results/figures/06b_시계열_{DOMAIN}_채.png", dpi=120, bbox_inches="tight")
print(f"저장 완료: ../results/figures/06b_시계열_{DOMAIN}_채.png")

genuine_df.to_csv(f"../results/reports/06b_조기탐지후보_{DOMAIN}_채.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: ../results/reports/06b_조기탐지후보_{DOMAIN}_채.csv")

저장 완료: ../results/figures/06b_시계열_IT기기_채.png
저장 완료: ../results/reports/06b_조기탐지후보_IT기기_채.csv
